# Embedding this repo for project / code guidance — token & cost sizing

**The question:** if I want to embed the Book Recommender (code + docs) so an assistant can
answer *"where does X live"* / *"how does this flow work"* questions, how many tokens is
that corpus, what does the index cost to build, and is building one even worth it?

How it counts:

- **File list comes from `git ls-files`**, so `.gitignore` has already done most of the
  filtering — `logs/`, `config/.env`, `*backup.sql`, `TODO.*` and `.venv` never appear.
- **Tokens come from `tiktoken`**, using the encoding of the embedding model this app
  already runs (`settings.openai.EMBEDDING_MODEL`; both `text-embedding-3-*` models use
  `cl100k_base`). Same helper shape as `evals/tools_catalog.py`.
- **Nothing calls OpenAI.** This is local counting only, so it is free to re-run as often
  as you like.

Read the totals as a *sizing estimate*, not a billing statement: chunkers split on
structure rather than exact token stride, so real chunk counts land within ~10% of these.

In [1]:
import json
import math
import subprocess
import sys
from pathlib import Path

import pandas as pd
import tiktoken

# The notebook lives at backend/evals/app_docs/, but should not care where the
# kernel was started from — walk up to the repo root, then put backend/ on the
# path so config.pricing imports the same way the eval reports do.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / ".git").exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("no .git found above cwd — start the kernel inside the repo")
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "backend"))
from config import settings                      # noqa: E402
from config.pricing import PRICES_CHECKED_ON, cost_of  # noqa: E402

EMBEDDING_MODEL = settings.openai.EMBEDDING_MODEL
CONFIGURED_DIMS = settings.openai.EMBEDDING_DIMENSIONS  # the app truncates the vector
ENCODER = tiktoken.encoding_for_model(EMBEDDING_MODEL)


def count_tokens(text: str) -> int:
    return len(ENCODER.encode(text))


GIT_SHA = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=REPO_ROOT, capture_output=True, text=True, check=True,
).stdout.strip()

print(f"repo      {REPO_ROOT}  @ {GIT_SHA}")
print(f"embedding {EMBEDDING_MODEL} @ {CONFIGURED_DIMS} dims "
      f"-> encoding {ENCODER.name}")
print(f"chat prices last verified {PRICES_CHECKED_ON} (config/pricing.py)")

repo      /home/tuani/Book-Recommender  @ 441d25e
embedding text-embedding-3-large @ 1024 dims -> encoding cl100k_base
chat prices last verified 2026-07-24 (config/pricing.py)


## 1. What goes in the corpus

`git ls-files` still hands back things that are worthless to embed for *code guidance*, so
two more filters run on top of it:

| Dropped | Why |
|---|---|
| `.csv`, `.jpg`, `.png`, `.svg`, `.ico` | binary / data, not guidance |
| `*.lock`, `package-lock.json` | dependency resolution noise — 90k tokens of it |
| `backend/evals/results/**` | per-campaign report + raw SQL dumps; regenerated output, and `CLAUDE.md` says never read the dumps |
| `backend/data/*.sql` | database backups |

Notebooks get one extra step: **only their `source` cells are counted, never their
outputs.** A single exploration notebook in `backend/data/` carries ~250k tokens of stored
output — more than the entire rest of the repo — and none of it describes how the app
works.

In [2]:
SKIP_EXTS = {".csv", ".jpg", ".jpeg", ".png", ".svg", ".ico", ".lock", ".log"}
SKIP_NAMES = {"package-lock.json"}
SKIP_PREFIXES = ("backend/evals/results/", "backend/logs/", "backend/data/backup.sql")

# Extensions worth embedding, plus extensionless files that carry real context.
TEXT_EXTS = {".py", ".md", ".jsx", ".js", ".css", ".html", ".json", ".yml", ".yaml",
             ".toml", ".sh", ".txt", ".sql", ".ipynb", ".example"}
TEXT_NAMES = {"Makefile", "makefile", "Dockerfile", "README", "LICENSE"}


def notebook_source(text: str) -> str:
    """A notebook stripped to its cell sources — outputs are stored execution
    noise (dataframes, plots, stack traces), not documentation of the app."""
    nb = json.loads(text)
    return "\n".join("".join(cell["source"]) for cell in nb["cells"])


def tracked_files() -> list[str]:
    out = subprocess.run(["git", "ls-files"], cwd=REPO_ROOT,
                         capture_output=True, text=True, check=True)
    return out.stdout.split("\n")


def area_of(rel_path: str) -> str:
    """Group a path into the area a reader would name: 'backend/app',
    'frontend/src', 'docs'. Root-level files collapse into '(root)'."""
    parts = rel_path.split("/")
    if len(parts) == 1:
        return "(root)"
    return parts[0] if parts[0] in {"docs", ".github"} else "/".join(parts[:2])


rows, skipped = [], []
for rel in tracked_files():
    if not rel:
        continue
    path = REPO_ROOT / rel
    ext = path.suffix.lower()

    if ext in SKIP_EXTS:
        skipped.append((rel, f"data/binary ({ext})"))
        continue
    if path.name in SKIP_NAMES or rel.startswith(SKIP_PREFIXES):
        skipped.append((rel, "generated/vendored"))
        continue
    if ext not in TEXT_EXTS and path.name not in TEXT_NAMES:
        skipped.append((rel, f"unrecognized ({ext or 'no ext'})"))
        continue
    try:
        text = path.read_text(encoding="utf-8")
    except (UnicodeDecodeError, FileNotFoundError) as exc:
        skipped.append((rel, type(exc).__name__))
        continue

    body = notebook_source(text) if ext == ".ipynb" else text
    rows.append({
        "path": rel,
        "area": area_of(rel),
        "ext": ext or path.name,
        "tokens": count_tokens(body),
        # what it would have cost to embed the file as-is, outputs and all
        "tokens_on_disk": count_tokens(text),
    })

files = pd.DataFrame(rows).sort_values("tokens", ascending=False, ignore_index=True)

print(f"counted {len(files)} files, skipped {len(skipped)}")
print(f"notebook outputs alone would have added "
      f"{files.tokens_on_disk.sum() - files.tokens.sum():,} tokens")
pd.Series([reason for _, reason in skipped]).value_counts().rename("skipped files")

counted 269 files, skipped 32
notebook outputs alone would have added 293,905 tokens


generated/vendored       18
data/binary (.svg)        5
unrecognized (no ext)     4
data/binary (.jpg)        2
data/binary (.csv)        1
data/binary (.log)        1
data/binary (.lock)       1
Name: skipped files, dtype: int64

## 2. Three corpus profiles

The whole repo is not one decision. **Core** is the app itself — what a guidance assistant
almost certainly needs. **Tests** and **playground** are large and useful for different
questions ("what is already covered?", "what did we try?"), so they are priced separately
rather than assumed in.

In [3]:
PROFILE_AREAS = {
    "playground": ["backend/playground"],
    "tests": ["backend/tests"],
    "data notebooks": ["backend/data"],
}
optional = [a for areas in PROFILE_AREAS.values() for a in areas]

CORE = "core (app + docs + evals + frontend)"
PROFILES = {
    CORE: ~files.area.isin(optional),
    "core + tests": ~files.area.isin(PROFILE_AREAS["playground"] +
                                     PROFILE_AREAS["data notebooks"]),
    "everything tracked": pd.Series(True, index=files.index),
}

profiles = pd.DataFrame({
    "files": {name: int(mask.sum()) for name, mask in PROFILES.items()},
    "tokens": {name: int(files.loc[mask, "tokens"].sum())
               for name, mask in PROFILES.items()},
})
profiles["avg tokens/file"] = (profiles.tokens / profiles.files).round().astype(int)
profiles

,files,tokens,avg tokens/file
core (app + docs + evals + frontend),193,170052,881
core + tests,226,217782,964
everything tracked,269,301328,1120


In [4]:
by_area = (files.groupby("area")
           .agg(files=("path", "size"), tokens=("tokens", "sum"))
           .sort_values("tokens", ascending=False))
by_area["% of repo"] = (100 * by_area.tokens / by_area.tokens.sum()).round(1)
by_area.head(15)

,files,tokens,% of repo
area,,,
backend/playground,39,79512,26.4
backend/tests,33,47730,15.8
backend/evals,13,39954,13.3
backend/app,51,31443,10.4
frontend/src,42,29929,9.9
backend/db,31,21349,7.1
docs,8,20735,6.9
backend/common,10,5914,2.0
(root),3,4382,1.5


In [5]:
by_ext = (files.groupby("ext")
          .agg(files=("path", "size"), tokens=("tokens", "sum"))
          .sort_values("tokens", ascending=False))

print("--- by file type ---")
print(by_ext.head(10).to_string())
print("\n--- 10 largest files in the corpus ---")
print(files.head(10)[["path", "tokens"]].to_string(index=False))

--- by file type ---
          files  tokens
ext                    
.py         152  127742
.json        11   57038
.md          24   36239
.jsx         26   19132
.txt          6   18480
.ipynb        5   15572
.sql         10    8143
.css          8    7177
.js          10    3631
Makefile      2    2781

--- 10 largest files in the corpus ---
                                                              path  tokens
        backend/playground/prompting/planner_payload_extended.json   11602
                   backend/playground/files/websearch_exmaple.json    9238
                             backend/evals/suites/query_suite.json    8730
                 backend/evals/suites/query_suite_adversarial.json    8696
                     backend/playground/notebook/book_entity.ipynb    8483
                 backend/playground/prompting/planner_payload.json    7874
                 backend/playground/prompting/planner_prompt_2.txt    7849
         backend/playground/prompting/planner_promp

## 3. Chunks, not files

You embed chunks, so the chunk count — not the token count — is what sizes the vector
store, and overlap means the tokens actually sent to the embedding endpoint exceed what
the repo holds.

`CHUNK_TOKENS` / `CHUNK_OVERLAP` below are the two knobs worth playing with. 800/100 is a
reasonable code-RAG default; smaller chunks retrieve more precisely and cost more to
store.

Embedding rates are **USD per 1M tokens**, transcribed from OpenAI's pricing page and
carrying the same warning `config/pricing.py` puts on the chat rates: *these go stale —
re-verify before trusting a number that matters.* They live here rather than in
`config/pricing.py` because the app never bills embeddings today; move them there if it
starts to.

In [6]:
CHUNK_TOKENS = 800
CHUNK_OVERLAP = 100

EMBEDDING_PRICES_CHECKED_ON = "2026-08-01"
EMBEDDING_PRICES = {                    # USD per 1M tokens
    "text-embedding-3-small": 0.02,
    "text-embedding-3-large": 0.13,
}
BYTES_PER_FLOAT = 4                     # float32 vectors, as stored by pgvector

# Index size is priced at the dimensions this app actually configures, not the
# model's native width: config/.env truncates the vector (1024), which is a
# supported and much cheaper way to store text-embedding-3-large.
INDEX_DIMS = CONFIGURED_DIMS


def chunks_for(tokens: int) -> int:
    """Chunk count for one file at the stride above. A file shorter than one
    chunk is still one chunk, never zero."""
    stride = CHUNK_TOKENS - CHUNK_OVERLAP
    return max(1, math.ceil(max(tokens - CHUNK_OVERLAP, 1) / stride))


files["chunks"] = files.tokens.map(chunks_for)

sizing_rows = []
for name, mask in PROFILES.items():
    subset = files.loc[mask]
    n_chunks = int(subset.chunks.sum())
    # every chunk past the first in a file re-sends CHUNK_OVERLAP tokens
    billed = int(subset.tokens.sum() + (n_chunks - len(subset)) * CHUNK_OVERLAP)
    row = {"tokens": int(subset.tokens.sum()), "chunks": n_chunks,
           "billed tokens": billed,
           "index MB": round(n_chunks * INDEX_DIMS * BYTES_PER_FLOAT / 1e6, 2)}
    for model, rate in EMBEDDING_PRICES.items():
        row[f"$ {model.replace('text-embedding-3-', '')}"] = round(
            billed / 1_000_000 * rate, 4)
    sizing_rows.append(row)

sizing = pd.DataFrame(sizing_rows, index=list(PROFILES))
print(f"chunks of {CHUNK_TOKENS} tokens, {CHUNK_OVERLAP} overlap | "
      f"index sized at {INDEX_DIMS} dims")
sizing

chunks of 800 tokens, 100 overlap | index sized at 1024 dims


,tokens,chunks,billed tokens,index MB,$ small,$ large
core (app + docs + evals + frontend),170052,345,185252,1.41,0.0037,0.0241
core + tests,217782,427,237882,1.75,0.0048,0.0309
everything tracked,301328,571,331528,2.34,0.0066,0.0431


## 4. Is the index worth building?

Building the index is cheap enough to be free in practice. The real cost is **per query**,
and that is where the comparison lives: retrieving the top *k* chunks versus simply pasting
the whole corpus into the prompt.

`gpt-5.6-luna` is the planner's current parse model; the cheaper rows show what the same
choice costs on a smaller model. The **cached** column matters most — a corpus that is
identical on every request is exactly what OpenAI's prompt cache is for, and this repo's
own eval reports already track cache hit rate for that reason.

In [7]:
TOP_K = 8                # chunks handed to the model per question
CORE_TOKENS = int(files.loc[PROFILES[CORE], "tokens"].sum())
RETRIEVED_TOKENS = TOP_K * CHUNK_TOKENS

comparison = []
for model in ["gpt-5.6-luna", "gpt-4.1", "gpt-4.1-mini", "gpt-5-mini"]:
    comparison.append(pd.Series({
        "whole corpus, uncached": cost_of(model, CORE_TOKENS, 0, 0),
        "whole corpus, cached": cost_of(model, CORE_TOKENS, CORE_TOKENS, 0),
        f"top-{TOP_K} chunks": cost_of(model, RETRIEVED_TOKENS, 0, 0),
    }, name=model))

print(f"core corpus {CORE_TOKENS:,} tokens vs {RETRIEVED_TOKENS:,} retrieved "
      f"({TOP_K} x {CHUNK_TOKENS})")
print("USD per request, input tokens only (no output):")
pd.DataFrame(comparison).map(lambda v: f"${v:.5f}" if v is not None else "unpriced")

core corpus 170,052 tokens vs 6,400 retrieved (8 x 800)
USD per request, input tokens only (no output):


,"whole corpus, uncached","whole corpus, cached",top-8 chunks
gpt-5.6-luna,$0.17005,$0.01701,$0.00640
gpt-4.1,$0.34010,$0.08503,$0.01280
gpt-4.1-mini,$0.06802,$0.01701,$0.00256
gpt-5-mini,$0.04251,$0.00425,$0.00160


In [8]:
# Break-even: how many questions before the index build pays for itself, versus
# pasting the core corpus into every prompt on the planner's model.
MODEL = "gpt-5.6-luna"
build_cost = float(sizing.loc[CORE, "$ large"])
retrieval_cost = cost_of(MODEL, RETRIEVED_TOKENS, 0, 0)

print(f"index build (core, text-embedding-3-large): ${build_cost:.4f}")
for label, stuffed in [("uncached", cost_of(MODEL, CORE_TOKENS, 0, 0)),
                       ("cached", cost_of(MODEL, CORE_TOKENS, CORE_TOKENS, 0))]:
    saved = stuffed - retrieval_cost
    if saved <= 0:
        print(f"  vs {label} corpus: retrieval is not cheaper — don't bother")
        continue
    print(f"  vs {label} corpus: saves ${saved:.5f}/question, "
          f"repaid by question #{max(1, math.ceil(build_cost / saved))}")

index build (core, text-embedding-3-large): $0.0241
  vs uncached corpus: saves $0.16365/question, repaid by question #1
  vs cached corpus: saves $0.01061/question, repaid by question #3


## 5. Reading the result

At the time of writing (2026-08-01, `441d25e`) the numbers came out as:

- **~301k tokens** for everything tracked, **~170k** for the core corpus (app, db, evals,
  config, common, docs, frontend source). Stripping notebook outputs and lockfiles removes
  roughly half the repo — the naive count is ~595k, and ~294k of that gap is stored
  notebook output alone.
- **345 chunks** for the core corpus at 800/100 — about **1.4 MB** of `float32` vectors at
  the 1024 dimensions this app configures. That is small enough to sit in the existing
  pgvector database beside the book embeddings with no new infrastructure.
- **Building the index costs about 2½ cents** on `text-embedding-3-large` (a third of a
  cent on `-small`). Build cost is not the reason to hesitate.
- **The core corpus fits in a modern context window.** With prompt caching, pasting all
  ~170k tokens into `gpt-5.6-luna` costs ~$0.017 per request against ~$0.006 for eight
  retrieved chunks — the same order of magnitude. The argument for retrieval here is
  *precision and latency*, not spend.

The largest single lever is scope, not chunk size: `backend/playground` and
`backend/tests` are ~42% of the tracked corpus and answer different questions than "how
does this app work". Decide those in or out before tuning anything else.

**Re-running:** all local, no API calls, no backend, no database — safe to re-run any time.
Numbers drift as the repo grows, so treat any figure pasted elsewhere as dated.